In [ ]:
import pandas as pd
import spacy
import os
import json
import pickle
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity


def sub_scr(model: str):
    path = os.getcwd()
    if path.endswith("Utils"):
        path = os.path.dirname(path)
        path = os.path.dirname(path)

    data_path = os.path.join(path, "Data", "raw", "Articles", "csv_format", "articles_train_gold.csv")
    test_set = pd.read_csv(data_path, sep='|')
    test_set = test_set[["pmid", "title", "abstract"]]
    test_set = test_set[:10]  # small test

    allowed_values = [
        "RoBERTa",
        "SciBERT",
        "BioBERT-base",
        "BioBERT-large",
        "BlueBERT-pubmed_NER",
        "BlueBERT-pubmed-mimic_NER"
    ]
    if model not in allowed_values:
        raise ValueError(f"{model} value not allowed. Allowed values: {allowed_values}")

    model_path = os.path.join(path, "Models", f"{model}_NER", "output", "model-best")
    trained_model = spacy.load(model_path)

    linking_model_path = os.path.join(path, "cfg")
    linking_model = SentenceTransformer(linking_model_path)

    with open(os.path.join(path, "DataFile", "kb_definitions.pickle"), "rb") as f:
        kb_definitions = pickle.load(f)

    with open(os.path.join(path, "DataFile", "kb_synonyms_labels.pickle"), "rb") as f:
        kb_synonyms_labels = pickle.load(f)


    entity_map = {}
    for entity_id, aliases in kb_synonyms_labels.items():
        definition = kb_definitions.get(entity_id, "")
        entity_map[entity_id] = definition

    entity_ids = list(entity_map.keys())
    contexts = list(entity_map.values())

    context_embeddings = linking_model.encode(contexts)

    def link_entity(mention):
        mention_emb = linking_model.encode(mention)
        scores = cosine_similarity([mention_emb], context_embeddings)[0]
        best_idx = np.argmax(scores)
        return entity_ids[best_idx], scores[best_idx]

    def id_to_uri(entity_id):
        return f"http://id.nlm.nih.gov/mesh/{entity_id}"

    result = {}

    for article in test_set.iterrows():
        entities = []
        print(f"Processing article {article[0]} out of {len(test_set)-1}")

        pmid = int(article[1]["pmid"])
        title = article[1]["title"]
        abstract = article[1]["abstract"]

        result[pmid] = {"entities": entities}

        nlp_title = trained_model(title)
        nlp_abstract = trained_model(abstract)

        for ent in nlp_title.ents:
            entity_id, score = link_entity(ent.text)
            uri = id_to_uri(entity_id)

            entities.append({
                "start_idx": ent.start_char,
                "end_idx": ent.end_char,
                "location": "title",
                "text_span": ent.text,
                "label": ent.label_,
                "uri": uri
            })

        for ent in nlp_abstract.ents:
            entity_id, score = link_entity(ent.text)
            uri = id_to_uri(entity_id)

            entities.append({
                "start_idx": ent.start_char,
                "end_idx": ent.end_char,
                "location": "abstract",
                "text_span": ent.text,
                "label": ent.label_,
                "uri": uri
            })
        
    output_path = os.path.join(path, "Output", "submission_NER.json")
    with open(output_path, "w") as f:
        json.dump(result, f, indent=4)

    return result


NER = sub_scr("RoBERTa")